### **DE_NB_LGL_REG_CREATE_SHORTCUTS**
___
###### **Purpose**: Creation of Legal and Regulatory Reporting Commercial Structure Shortcuts

###### **Sources (input tables)**: 


###### **Sink Tables (outputs)**: 


###### **Parameters**:

___
###### **Changelog**:
- yyyy-mm-dd | user | Description
- 2026-11-18 | Laura Pellicer | DPM engine: DataLoader raw tables (raw_dpm_param_*, raw_dpm_exec_req)
- 2026-11-18 | Laura Pellicer | DPM engine: FCP source tables from DP {env} Funding Capital Plan Processing

###### **Technical debt**
- table_example_AB - technical debt - description

###### **Relevant notes**
- ER Studio model: **Not yet created.**
- Table definition based on mapping: **[Mapping_DA2_LegalAndRegulatoryReporting](link)**.

___

#### **Session Configuration**
___

In [1]:
%%configure -f
{
    "driverMemory": "28g",
    "driverCores": 4,
    "executorMemory": "28g",
    "executorCores": 4,
    "conf": {
        "spark.native.enabled": "true",
        "spark.dynamicAllocation.maxExecutors": "2"
    }
}

StatementMeta(, 92355e5b-80d0-438c-9b6d-46f8eca08d2c, -1, Finished, Available, Finished, True)

<div style="font-family:monospace;font-size:13px">
    <p>
        <span style="color:red;font-weight:bold">Warning:</span> When you want to configure the driverMemory, driverCores, executorCores,
        executorMemory, or numExecutors, check the workspace settings and default pool node details from workspace settings -> Default pool
        for workspace -> Pool details, you can use %%configure set customized compute configurations that within the range of Pool level size.
    </p>
    <p>
        Notice that if the session configuration is different from the live session (8 vCores, 56GB memory, 2 executors), the session
        start will fall back to the on-demand session which will take about 3~5 mins to spin up.
    </p>
    <p>The Executor/Driver size buckets on Fabric are: </p>
    <ul>
        <li>A small size bucket: 4 vCores/28 GB memory</li>
        <li>A medium size bucket: 8 vCores/56 GB memory</li>
        <li>A large size bucket: 16 vCores/112 GB memory</li>
        <li>A xlarge size bucket: 32 vCores/224 GB memory</li>
        <li>A xxlarge size bucket: 64 vCores/400 GB memory</li>
    </ul>
    <p>
        We recommend avoiding using Driver or Executor vCore/Memory counts that are not in the size bucket specified above, so be sure
        to specify the <b>Driver and Executor</b> vCore/Memory values in relation to the above size buckets (4vCores/28GB, 8vCores/56GB,
        16vCores/112GB, 32vCores/224GB, or 64vCores/400GB), moreover, the executor size and driver size should be the same.
    </p>
</div>

#### **Imports**
___

In [1]:
from workerFactory import workerfactory
import medallion_framework
import sempy.fabric as fabric
import re
import requests
import json
from collections import defaultdict

StatementMeta(, f0ebd26e-5ada-4a58-887e-104d9f807838, 5, Finished, Available, Finished, False)

<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead


#### **Procedures**
___

In [2]:
token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
client = fabric.FabricRestClient()
current_ws_name = fabric.resolve_workspace_name()
current_env = "FET" if " FET " in current_ws_name else "INT" if " INT " in current_ws_name else "DEV" if " DEV " in current_ws_name else "QUA" if " QUA " in current_ws_name else "PRD"
fallback_env = "DEV"
wsdest = fabric.get_workspace_id()
lhdest = fabric.get_lakehouse_id()

def env_shortcuts(original_ws_name, list_tbl, list_tbl_env):
    env_by_tbl = dict(list_tbl_env)
    tables_by_env = {}

    for tbl in list_tbl:
        env = env_by_tbl.get(tbl)  # None if not found
        if env is None: env = current_env
        tables_by_env.setdefault(env, []).append(tbl)

    for env, tables in tables_by_env.items():
        if env == "FET":
            try:
                wsori  = fabric.resolve_workspace_id(original_ws_name.format(env=env))
            except Exception as e:
                wsori  = fabric.resolve_workspace_id(original_ws_name.format(env=fallback_env))
        else:
            wsori  = fabric.resolve_workspace_id(original_ws_name.format(env=env))
        lhori  = client.get(f"/v1/workspaces/{wsori}/lakehouses").json()["value"][0]["id"]
        post_shortcuts(tables, wsori, lhori, wsdest, lhdest)

def post_shortcuts(table_list, workspace_origem, lakehouse_origem, workspace_destino, lakehouse_destino):
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_destino}/items/{lakehouse_destino}/shortcuts?shortcutConflictPolicy=CreateOrOverwrite"
    
    shortcuts_ok = []
    shortcuts_err = []

    for table in table_list:
        shortcut_payload = {
            "name": table,
            "path": "Tables",
            "target": {
                "oneLake": {
                    "workspaceId": workspace_origem,
                    "itemId": lakehouse_origem,
                    "path": f"Tables/{table}"
                }
            }
        }

        try:
            response = requests.post(url, headers=headers, json=shortcut_payload)
            if response.status_code in (200,201,409):
                shortcuts_ok.append(table)
            else:
                data = json.loads(response.text)
                # Extract only the 'moreDetails' section
                data = data.get("moreDetails", [])
                cleaned_msg = json.dumps(data, ensure_ascii=False)
                shortcuts_err.append((table, cleaned_msg))
        except requests.exceptions.RequestException as e:
            shortcuts_err.append((table, str(e)))

    # Mostrar shortcuts criados com sucesso
    if shortcuts_ok:
        print("\n✅ Shortcuts criados com sucesso:")
        print(", ".join(shortcuts_ok))

    # Mostrar erros agrupados
    if shortcuts_err:
        print("❌ Shortcuts com erro:")
        grouped_errors = defaultdict(list)
        for t, msg in shortcuts_err:
            grouped_errors[msg].append(t)
        for msg, tables in grouped_errors.items():
            print(f"🔴 {msg}:")
            print(", ".join(tables))

def drop_table(table_name):
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    print(f"Dropped {table_name}")

def drop_all_tables_show_tables(dry_run: bool = True):
    target_preffixes = ("tmp_", "stag_", "staging_", "stg_")
    target_suffixes = ("_prd", "_qua", "_dev", "_old")
    current_catalog = spark.sql("SELECT current_catalog()").first()[0]
    current_db      = spark.sql("SELECT current_database()").first()[0]

    show_df = spark.sql(f"SHOW TABLES IN `{current_catalog}`.`{current_db}`")
    # Columns are typically: database, tableName, isTemporary
    names = [r["tableName"] for r in show_df.toLocalIterator()]

    if dry_run:
        print(f"Dry run — would drop from `{current_catalog}`.`{current_db}`:")
        for n in names:
            if n and n.lower().endswith(target_suffixes):
                print(f"  - `{current_catalog}`.`{current_db}`.`{n}`")
            elif n and n.lower().startswith(target_preffixes):
                print(f"  - `{current_catalog}`.`{current_db}`.`{n}`")
        print(f"Total: {len(names)}")
        return

    for n in names:
        if n and n.lower().endswith(target_suffixes):
            fq = f"`{current_catalog}`.`{current_db}`.`{n}`"
            drop_table(fq)
        elif n and n.lower().startswith(target_preffixes):
            fq = f"`{current_catalog}`.`{current_db}`.`{n}`"
            drop_table(fq)

StatementMeta(, f0ebd26e-5ada-4a58-887e-104d9f807838, 6, Finished, Available, Finished, False)

In [3]:
drop_all_tables_show_tables(False)

StatementMeta(, f0ebd26e-5ada-4a58-887e-104d9f807838, 7, Finished, Available, Finished, False)

#### **Shortcuts**
___

In [6]:
current_ws_name = fabric.resolve_workspace_name()
list_tbl_env = []

# Accounting
list_tbl = ['enr_rlt_acg_idt_idc_cmb', 'enr_rlt_acg_idt_tmplt_idc', 'enr_acg_csl_ldgr', 'enr_int_acg_bal_dt', 'enr_acg_cntrct_ldgr']
env_shortcuts('ENR {env} Accounting', list_tbl, list_tbl_env)

# Reference Data
list_tbl = ['enr_lh_cdelst', 'enr_lh_cdelst_att', 'enr_exh']
env_shortcuts('ENR {env} Reference Data', list_tbl, list_tbl_env)

# Commercial Structure
list_tbl = ['enr_ent', 'enr_rlt_ent']
env_shortcuts('ENR {env} Commercial Structure', list_tbl, list_tbl_env)

# RAW
list_tbl = ['raw_ORA_DIF_DOMINFO_FP_M_CARTEIRA_PROPRIA_OUT_VW']
env_shortcuts('RAW {env} Lakehouse', list_tbl, list_tbl_env)

# Party
list_tbl = ['enr_prty','enr_prty_doc','enr_rlt_prty_ctry','enr_rlt_prty_bkg_ent','enr_adrs','enr_rlt_prty_adrs','enr_prty_ind']
env_shortcuts('ENR {env} Party', list_tbl, list_tbl_env)

# Instruments
list_tbl = ['enr_instrmnt', 'enr_cntrct', 'enr_rlt_instrmnt_prty', 'enr_instrmnt_fin_pos']
env_shortcuts('ENR {env} Instruments', list_tbl, list_tbl_env)

# Risk Parameters
list_tbl = ['enr_prty_rsk_sgm']
env_shortcuts('ENR {env} Risk Parameters', list_tbl, list_tbl_env)

# DataLoad
list_tbl = ['RAW_LGL_REG_DIM_RPG_ENT']
env_shortcuts('RAW {env} DATALOAD', list_tbl, list_tbl_env)

# DataLoad - DPM engine: parameterisation file (four tabs) and execution requests
list_tbl = ['raw_dpm_param_datapoint', 'raw_dpm_param_fltr', 'raw_dpm_param_ctlg', 'raw_dpm_param_src_tbl', 'raw_dpm_exec_req']
env_shortcuts('RAW {env} DATALOAD', list_tbl, list_tbl_env)

# Funding Capital Plan - DPM engine: FCP source tables
list_tbl = ['dp_fcp_bdp_global_datapoints', 'raw_fcp_gap_datapoints']
env_shortcuts('DP {env} Funding Capital Plan Processing', list_tbl, list_tbl_env)

# Capital Markets
list_tbl = ['enr_cap_mkt_instrmnt', 'enr_cap_mkt_opt']
env_shortcuts('ENR {env} Capital Market', list_tbl, list_tbl_env)

# Banking Assets
list_tbl = ['enr_instrmnt_assts', 'enr_instrmnt_assts_prc']
env_shortcuts('ENR {env} Banking Assets', list_tbl, list_tbl_env)

# Banking Securities
list_tbl = [ 'enr_sec', 'enr_instrmnt_sec_hld','enr_sec_inl_fin_acc']
env_shortcuts('ENR {env} Banking Securities', list_tbl, list_tbl_env)

StatementMeta(, f0ebd26e-5ada-4a58-887e-104d9f807838, 10, Finished, Available, Finished, False)

WorkspaceNotFoundException: Workspace 'ENR DEV Accounting' not found

#### **Engine Execution**
___

In [10]:
#Engine setup
engine = workerfactory.WorkerFactory(spark,mssparkutils,medallion_framework,verbose=False)
engine.send_finish_notification()

StatementMeta(, 92355e5b-80d0-438c-9b6d-46f8eca08d2c, 13, Finished, Available, Finished, False)

2026-11-12 11:44:11,432 - WorkerFactory - send_finish_notification - ⚠️ WARNING - ⚠️ Environment 'FET' does not have Orchestration. Skipping 'AgentEvent.End' notification.)
